# Preprocessing on Primary Land Use Tax Lot Output (PLUTO) Dataset:

----

## Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("preprocessing_pluto")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

## Read PLUTO Parquet File:

In [ ]:
base_dir = "../data"
pluto_path = base_dir + '/raw/pluto/pluto.parquet'
pluto_sdf = spark.read.parquet(pluto_path)

In [ ]:
pluto_sdf.printSchema()

In [ ]:
# Check the shape of parquet file
num_rows = pluto_sdf.count()
print(f"Number of rows: {num_rows}")

columns = pluto_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
pluto_sdf.show(5)

## Drop Unrelated Columns:

In [ ]:
# calculate the amount of NULL in each column
na_counts = pluto_sdf.select([sum(col(column).isNull().cast("int")).alias(column) for column in pluto_sdf.columns])
na_counts.show()

Leave only the helpful columns:

In [ ]:
pluto_sdf = pluto_sdf.select('borough', 'bldgclass', 'latitude', 'longitude')
pluto_sdf.show(5)

## Data Cleaning:

Rename columns for more clear:

In [ ]:
pluto_sdf = pluto_sdf.withColumnRenamed("bldgclass", "building_class")
pluto_sdf.show(5)

Based on the PLUTO data dictionary, column `building_class` specify the general category and specific categery of each building unit. For example, under the general class code K. Store Buildings (Taxpayers Included), it has K6-Shopping Centers with or without Parking and K7-Banking Facilities with or without Parking.

For our research purpose, we are mainly focus on general class of the building (e.g. "K" instead of "K6" or "K7"), so we will only keep the first character (general class) of the building class codes.

In [ ]:
pluto_sdf = pluto_sdf.withColumn('building_class', 
                                 substring(col('building_class'), 1, 1))
pluto_sdf.show(5)

# Save the Preprocessed PLUTO Dataset:

In [ ]:
pluto_dir = base_dir + '/curated/pluto'
file_name = 'preprocessed_pluto'
pluto_path = os.path.join(pluto_dir, file_name)
pluto_sdf.write.mode('overwrite').parquet(pluto_path)